In [11]:
import numpy as np

data = np.array([
    [1,0,1,0,0],
    [0,1,0,1,1],
    [1,0,0,0,0],
    [0,1,1,1,1],
    [1,0,1,0,0],
    [0,1,0,1,1],
])

labels = np.array([0,1,0,1,0,1])

train_data = data[:4]
train_labels = labels[:4]
test_data = data[4:]
test_labels = labels[4:]

def calc_prior(labels):
    n = len(labels)
    
    ps = {}
    for label in np.unique(labels):
        print(labels == label)
        ps[label] = np.sum(labels == label) / n
        
    return ps

prior = calc_prior(train_labels)

def calc_like(data, labels):
    n = len(labels)
    
    ps = {}
    for label in np.unique(labels):
        label_data = data[labels == label]
        p = (np.sum(label_data, axis=0) + 1) / (n+1)
        ps[label] = p

    return ps

likelihood = calc_like(train_data, train_labels)

def predict(sample, prior, likelihood):
    log_probs = {}
    
    for label in np.unique(labels):
        log_pro = np.log(prior[label])
        
        for i in range(len(sample)):
            if sample[i] == 1:
                log_pro += np.log(likelihood[label][i])
            else:
                log_pro += np.log(1 - likelihood[label][i])
                
        log_probs[label] = log_pro
        
    return max(log_probs, key=log_probs.get)

for sample in test_data:
    print(predict(sample, prior, likelihood))

[ True False  True False]
[False  True False  True]
0
1


In [30]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
data = iris.data
labels = iris.target

# train_data = data[:120]
# train_labels = labels[:120]
# test_data = data[120:]
# test_labels = labels[120:]

train_data, test_data, train_labels, test_labels = train_test_split(data, labels, test_size=0.2, random_state=10)


def cal_prior(labels):
    ps = {}

    for cls in np.unique(labels):
        ps[cls] = np.sum(labels == cls) / len(labels)

    return ps

prior = cal_prior(train_labels)

def calc_mean_var(data, labels):
    mean_var = {}
    
    for label in np.unique(labels):
        class_data = data[label == labels]
        mean_var[label] = {'mean': np.mean(class_data, axis=0), 'var': np.var(class_data, axis=0)}
        
    return mean_var

mean_var = calc_mean_var(train_data, train_labels)

def pdf(x, mean, var):
    return 1 / np.sqrt(2 * np.pi * var) * np.exp(-((x - mean)**2 / (2 * var)))

def predict(sample, prior, mean_var):
    log_probs = {}
    
    for cls in prior:
        log_prob = np.log(prior[cls])
        
        for i in range(len(sample)):
            mean = mean_var[cls]['mean'][i]
            var = mean_var[cls]['var'][i]
            
            log_prob += np.log(pdf(sample[i], mean, var))
            
        log_probs[cls] = log_prob
    
    return max(log_probs, key=log_probs.get)

predictions = []
for sample in test_data:
    predictions.append(predict(sample, prior, mean_var))
    
print(predictions == test_labels)

[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True]
